# 1. Filtering, grouping, and merging: preserving meaning in labeled tables

Learn to select observations with pandas, calculate group summaries, and join reference data without accidentally multiplying rows. Prerequisite: [pandas basics](16_pandas_basics.ipynb), conditions, and dictionaries. The synthetic dataset contains four invented orders. It is deliberately small enough to inspect every intermediate table by hand. You will distinguish an order, a store, and a group summary as different observational units.

## 2. What problem does this solve?

Data rarely arrive as one perfectly arranged table. Orders may be stored separately from store information. Analysts select rows, summarize groups, and combine tables. A join can unexpectedly duplicate observations when its keys are not unique. The resulting row count may look plausible while totals double and later models train on duplicated examples.

Our task is to summarize paid orders and attach store regions. We will check counts, totals, key uniqueness, missing matches, and source preservation. These checks establish table integrity for this example, not the correctness of any business decision or prediction model.

## 3. Intuition and analogy

Filtering resembles selecting index cards that meet a rule. Grouping places cards in labeled piles, then calculates one summary per pile. Merging attaches information from another set of cards using a shared key.

If a store has two reference cards, one order may match both and appear twice. If no store card exists, a left join keeps the order but marks the new information missing. These are consequences of the matching rule, not random software behavior. Decide which kinds of matches are allowed before merging.

## 4. Terminology

A **Boolean filter** keeps rows where a condition is true. `.loc` selects by labels or a Boolean mask; `.iloc` selects by integer position. A **group key** identifies a pile of observations. An **aggregation** reduces a group to a summary. A **join key** links tables. **Cardinality** describes how many rows on each side may share a key.

A **many-to-one** join allows many orders per store but one reference row per store. A **left join** retains all left-side observations. An **inner join** retains matched observations only. A **missing match** is not the same as a missing value already present in a matched reference row. `indicator=True` records match status so that these cases can be investigated separately.

## 5. Mathematical foundations

For a store $g$, let $I_g$ be the set of paid-order row indices for that store. Define

$$n_g=|I_g|,\qquad R_g=\sum_{i\in I_g}r_i,\qquad \bar r_g=R_g/n_g.$$

$r_i$ is revenue for order $i$ in dollars. $n_g$ is the order count, $R_g$ is total revenue in dollars, and $\bar r_g$ is mean dollars per order. The notation $|I_g|$ means the number of indices in the set. The mean requires $n_g>0$.

For a valid many-to-one left join with no omitted left rows, output row count should equal input row count. For disjoint store groups, their counts sum to the number of paid orders and their revenue totals sum to paid revenue. These are conservation checks; they do not prove the original measurements are truthful.

## 6. Hand-calculated example

Orders 101 and 102 are paid store-A orders worth 10 and 30 dollars. Order 103 is an unpaid store-B order worth 20, and order 104 is a paid store-B order worth 40. Filtering paid orders retains IDs 101, 102, and 104 with total $10+30+40=80$ dollars.

Store A has two paid orders, total 40 dollars and mean 20 dollars per order. Store B has one paid order, total 40 dollars and mean 40 dollars per order. Averaging the two group means equally gives 30, but the overall paid-order mean is $80/3\approx26.67$. Unequal group sizes explain the difference.

## 7. Implementation from scratch

Use a dictionary keyed by store to accumulate count and revenue. Each paid order contributes once. This loop exposes what a later `groupby` call computes. A group summary has one row per store, unlike the original order table's one row per order.

In [ ]:
# Enable embedded figures in this fresh notebook kernel.
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

In [ ]:
records = [
    {'order_id': 101, 'store': 'A', 'revenue': 10.0, 'paid': True},
    {'order_id': 102, 'store': 'A', 'revenue': 30.0, 'paid': True},
    {'order_id': 103, 'store': 'B', 'revenue': 20.0, 'paid': False},
    {'order_id': 104, 'store': 'B', 'revenue': 40.0, 'paid': True},
]
grouped = {}
for record in records:
    if record['paid']:
        group = grouped.setdefault(record['store'], {'count': 0, 'total': 0.0})
        group['count'] += 1
        group['total'] += record['revenue']
for group in grouped.values():
    group['mean'] = group['total'] / group['count']
assert grouped == {'A': {'count': 2, 'total': 40.0, 'mean': 20.0},
                   'B': {'count': 1, 'total': 40.0, 'mean': 40.0}}
print('Scratch paid-order summary:', grouped)

## 8. Inspecting row selection

In [ ]:
import pandas as pd
import numpy as np
orders = pd.DataFrame(records)
source_snapshot = orders.copy(deep=True)
assert orders['order_id'].is_unique
paid = orders.loc[orders['paid'], ['order_id', 'store', 'revenue']].copy()
assert paid['order_id'].tolist() == [101, 102, 104]
assert paid.shape == (3, 3)
assert paid['revenue'].sum() == 80.0
print(paid.to_string(index=False))
print('Positional first row ID:', orders.iloc[0]['order_id'])

The mask is computed from the same table it selects. A Boolean Series carries index labels; using a mask built from a differently indexed table can align unexpectedly or fail. Keep the mask and table related explicitly. `.copy()` expresses that the derived table may be edited independently; avoid chained assignment.

## 9. pandas implementation

Named aggregation gives readable output columns. `size` counts rows; `count` on a column counts nonmissing values. They agree here because revenues are present, but answer different questions when measurements are missing. `dropna=False` keeps missing group keys visible rather than silently omitting their observations. We validate revenues as finite before summing.

In [ ]:
assert np.isfinite(paid['revenue']).all()
summary = paid.groupby('store', as_index=False, dropna=False).agg(
    order_count=('order_id', 'size'),
    total_revenue=('revenue', 'sum'),
    mean_revenue=('revenue', 'mean'),
)
assert summary['order_count'].sum() == len(paid)
assert summary['total_revenue'].sum() == paid['revenue'].sum()
assert summary.set_index('store')['mean_revenue'].to_dict() == {'A': 20.0, 'B': 40.0}
weighted_mean = (summary['mean_revenue'] * summary['order_count']).sum() / summary['order_count'].sum()
assert np.isclose(weighted_mean, paid['revenue'].mean())
print(summary.to_string(index=False))

## 10. Visualization

Show totals and counts separately. Equal store revenue does not mean equal order volume or equal order value. The two panels have different units and deliberately share only their store labels. The figure describes four invented orders after one has been excluded as unpaid.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].bar(summary['store'], summary['total_revenue'])
axes[0].set(ylabel='Paid revenue (dollars)', title='Equal totals')
axes[1].bar(summary['store'], summary['order_count'], color='darkorange')
axes[1].set(ylabel='Number of paid orders', title='Different counts', yticks=[0, 1, 2])
for ax in axes:
    ax.set_xlabel('Store')
fig.tight_layout()
display(fig)
plt.close(fig)

## 11. Experiment: safe joins and rejected cardinality

Attach one region per store with a many-to-one left join. `validate='many_to_one'` rejects duplicate keys on the reference side rather than accepting a row multiplication. An explicit join key prevents accidental matching on other identically named columns. After joining, verify identity and total preservation.

In [ ]:
stores = pd.DataFrame({'store': ['A', 'B'], 'region': ['North', 'South']})
assert stores['store'].notna().all() and stores['store'].is_unique
joined = paid.merge(stores, on='store', how='left', validate='many_to_one', indicator=True)
assert len(joined) == len(paid)
assert joined['order_id'].tolist() == [101, 102, 104]
assert joined['_merge'].eq('both').all()
assert joined['revenue'].sum() == 80.0
print(joined.to_string(index=False))

duplicate_reference = pd.concat([stores, stores.iloc[[0]]], ignore_index=True)
try:
    paid.merge(duplicate_reference, on='store', how='left', validate='many_to_one')
except pd.errors.MergeError:
    print('Duplicate reference keys correctly rejected.')
else:
    raise AssertionError('Invalid reference cardinality was accepted.')

incomplete_reference = stores.loc[stores['store'].eq('A')]
unmatched = paid.merge(incomplete_reference, on='store', how='left', validate='many_to_one', indicator=True)
assert unmatched.loc[unmatched['_merge'].eq('left_only'), 'order_id'].tolist() == [104]
pd.testing.assert_frame_equal(orders, source_snapshot)

## 12. Configuration choices

Filter rules, group keys, aggregation functions, and join cardinality are analysis choices. They are not automatically fitted model hyperparameters. A group statistic becomes learned feature state if you estimate it from training records and apply it to future observations. In that use, fit on training data only and define a fallback for unseen groups; a full-dataset target average can leak labels.

## 13. Evaluation

Check original key uniqueness, selected IDs, row counts, group count conservation, revenue conservation, expected mean values, join match status, and source preservation. A many-to-one join can still attach the wrong region if the reference data themselves are incorrect. Integrity tests establish structural behavior, while source audits establish meaning.

Missing observations need explicit policy. pandas sums may skip missing values; a total based on incomplete revenue is not a complete total. Pair row counts with nonmissing counts and report data coverage. Our finite-revenue assertion prevents that ambiguity in this example.

## 14. Assumptions and limitations

Every order ID is unique, each order belongs to one store, and all revenue values use dollars under the same accounting definition. Store reference keys are nonmissing and unique. pandas may match null join keys to each other, which can surprise readers expecting SQL-like null behavior; validate keys rather than relying on that behavior. These examples are in-memory and contain no private customer information.

## 15. Common mistakes and debugging

Do not drop duplicates merely to make a join run: investigate why multiple reference rows exist. They might represent history that requires a time-aware key. Do not choose an inner join just to remove missing matches without reporting the lost observations. Do not average group means equally when the desired unit is one original observation and group sizes differ.

Avoid chained assignments such as editing a filtered expression in one statement. Use `.loc` for deliberate source editing or an explicit derived copy. When totals change unexpectedly, inspect key multiplicities and match status before changing aggregation formulas.

## 16. Alternatives

A dictionary loop is transparent for a tiny example. pandas combines labeled tables and validation options. SQL is useful for persisted relational data and larger queries. NumPy is efficient for numerical arrays but does not supply named relational joins. The representation can change, but observational units, cardinality, missingness, and conservation rules remain relevant.

## 17. Exercises

- **Beginner:** Predict the paid-order IDs and total revenue before running the filter.
- **Beginner:** Explain why store B's mean paid revenue is 40 rather than 30.
- **Beginner:** Distinguish a row count from a nonmissing-value count.
- **Intermediate:** Reconstruct the overall paid-order mean from group means and counts.
- **Intermediate:** Create a reference missing store B and identify unmatched orders without dropping them.
- **Challenge:** Write a validated store-region attachment function that rejects missing or duplicate reference keys, retains every order, and reports unmatched IDs. Verify both a valid and an invalid reference.

## 18. Detailed solutions

The paid IDs are 101, 102, and 104, totaling 80 dollars. Store B's unpaid 20-dollar order is excluded, so its paid mean is 40. Row count counts observations regardless of a field's missingness; nonmissing count counts available measurements of that field. The weighted mean is $(20\times2+40\times1)/3=80/3$, not $(20+40)/2$.

With only store A in the reference, order 104 remains but has match status `left_only`. The challenge should inspect reference keys before merging and validate cardinality during the join. It should not invent a region for an unmatched store.

In [ ]:
def attach_regions(order_table, reference):
    if reference['store'].isna().any() or not reference['store'].is_unique:
        raise ValueError('Reference store keys must be present and unique.')
    if order_table['store'].isna().any():
        raise ValueError('Order store keys must be present.')
    result = order_table.merge(reference, on='store', how='left', validate='many_to_one', indicator=True)
    if len(result) != len(order_table):
        raise AssertionError('Join changed observation count.')
    unmatched_ids = result.loc[result['_merge'].eq('left_only'), 'order_id'].tolist()
    return result, unmatched_ids

complete, missing_ids = attach_regions(paid, stores)
assert not missing_ids and len(complete) == 3
partial, missing_ids = attach_regions(paid, incomplete_reference)
assert missing_ids == [104] and len(partial) == 3
assert np.isclose(weighted_mean, 80 / 3)
for bad_reference in [duplicate_reference, pd.DataFrame({'store': [None], 'region': ['Unknown']})]:
    try:
        attach_regions(paid, bad_reference)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid reference keys must fail.')
pd.testing.assert_frame_equal(orders, source_snapshot)
print('Weighted summaries, unmatched IDs, and reference validation passed.')

The helper assumes the documented columns and unique order IDs have already been validated. A general production join utility would need schema, column-collision, temporal, and data-type policies as well.

## 19. Knowledge check

**What is the unit of a grouped row?** One group, such as a store, rather than one original order.

**Why can a join multiply rows?** A left key can match multiple reference rows.

**What does a left join preserve?** Left-side observations, including those without a match.

**Why inspect the merge indicator?** It separates matched rows from missing key matches for auditing.

**Why weight group means?** Unequal group sizes contribute unequal numbers of observations to the overall mean.

## 20. Interview preparation

**How do you protect a many-to-one join?** Validate nonmissing unique reference keys, specify the join key and cardinality, then check row identity and counts.

**Why can missing group keys change totals?** Grouping may omit them unless explicitly retained; conservation checks reveal the loss.

**How does group feature leakage occur?** A statistic uses validation/test labels or future records when it should be fitted only on training information.

**Why is dropping duplicate reference rows risky?** The duplicates may represent legitimate history or inconsistent definitions requiring investigation.

**How do you debug doubled revenue?** Inspect join key multiplicities and row counts before assuming the revenue calculation is wrong.

## 21. Summary and next steps

Filtering changes which observations remain; grouping changes the observational unit; merging attaches information under a cardinality contract. Track identities and totals throughout. Continue to [visualizing data](18_visualizing_data.ipynb). Dedicated plotting and exploratory-analysis lessons are specified in the [module plan](README.md#01-19) until authored.